In [ ]:
import os
import time
import datetime
import matplotlib.pyplot as plt
import numpy as np
import pathlib as pl
import shutil
import sys
import pandas as pd

import flopy
from modflowapi import ModflowApi
from modflowapi.extensions import ApiSimulation

from bmi.wrapper import BMIWrapper

import pyswmm
from pyswmm import Simulation, Nodes
from pyswmm import Output

In [ ]:
sys.path.append("../common")
from liss_settings import \
    libmf6, \
    get_dflow_grid_name, get_dflow_dtuser, \
    get_modflow_coupling_tag, get_modflow_grid_name, \
    silent, verbosity, \
    print_path, print_value

# Load Names

## dflow

In [ ]:
control_path = pl.Path("../dflow-fm/highres/tides_atm_surge_2018/FlowFM.mdu") # change this if using a different D-Flow FM control file
grid_name = get_dflow_grid_name(control_path)
print(grid_name)
dflowfm_dtuser = get_dflow_dtuser(control_path)
print(f'{dflowfm_dtuser} seconds')



## modflow

In [ ]:
# mf_grid_name = get_modflow_grid_name()

#  Unit Conversions

In [ ]:
d2sec = 24. * 60. * 60.
hrs2sec = 60. * 60. 
m2ft = 3.28081
cfd2cms = 1.0 / ((m2ft**3) * 86400.)

# MODFLOW coupling frequency

Change the `mf_couple_freq_hours` value. Only tested for multiple of the D-Flow FM DtUser variable. Will not work for `mf_couple_freq_hours` values greater than 24.

In [ ]:
# mf_couple_freq_hours = 0.25 # Change this value to change the coupling frequency
# mf_couple_freq = mf_couple_freq_hours * hrs2sec
# dflow_per_mf = int(mf_couple_freq / dflowfm_dtuser)
# print(f"MODFLOW coupling frequency {mf_couple_freq_hours} hours\nMODFLOW coupled to D-FLOW FM every {dflow_per_mf} output time step ({dflowfm_dtuser} sec.)") 

# mf_tag = get_modflow_coupling_tag(mf_couple_freq_hours)
# print(f"MODFLOW coupling tag: {mf_tag}")

# mf_couple_nstp = int(86400.0 / (dflow_per_mf * dflowfm_dtuser))
# print(f"MODFLOW time steps per day: {mf_couple_nstp }")

# Set a few variables for controlling coupling

In [ ]:
HDRY = -1e30
DEPTH_MIN = 0.1

#### Print the path of the modflow6 shared library

In [ ]:
str(libmf6), libmf6.is_file()

# D-FLOW to MODFLOW weights


## GHB weights

In [ ]:
# fpath = f"../mapping/PJ/dflow_{grid_name}_to_{mf_grid_name}_ghb.npz"
# npzfile = np.load(fpath)
# print(fpath)
# dflow2mfghb = npzfile["dflow2mfghb"]
# print(f'dflow2ghb shape: {dflow2mfghb.shape}')
# ghbmask = npzfile["ghbmask"]
# print(f'ghb mask shape :{ghbmask.shape}')
# ghb2qext = npzfile["ghb2qext"]
# print(f'ghb2qext shape: {ghb2qext.shape}')

## CHD weights

In [ ]:
# fpath = f"../mapping/PJ/dflow_{grid_name}_to_{mf_grid_name}_chd.npz"

# print(fpath)
# npzfile = np.load(fpath)
# dflow2mfchd = npzfile["dflow2mfchd"]
# print(f'dflow2chd shape: {dflow2mfchd.shape}')
# chdmask = npzfile["chdmask"]
# print(f'chd mask shape :{chdmask.shape}')
# chd2qext = npzfile["chd2qext"]
# print(f'chd2qext shape: {chd2qext.shape}')

# Create Modflow Model

In [ ]:
# # set path to gwf model (uncoupled)
# mf_base_path = pl.Path("../modflow/pj_2018_adjust_FINAL/base/").resolve()
# # create directory to run coupled gwf model
# mf_run_path = pl.Path(f"../modflow/pj_2018_adjust_FINAL/MF+DFlow/run_{mf_tag}/").resolve()


In [ ]:
# sim = flopy.mf6.MFSimulation.load(sim_ws=mf_base_path, verbosity_level=verbosity())
# gwf = sim.get_model()

In [ ]:
# sim.set_sim_path(mf_run_path)
# # create outputs in mf_run_path
# (mf_run_path / 'outputs').mkdir(exist_ok=True)

## Write the new model files

In [ ]:
# sim.write_simulation(silent=silent())

# Define base GHB variables

In [ ]:
# ghb_data0 = gwf.ghb.stress_period_data.get_dataframe()[0]
# assert ghb_data0.shape[0] == ghbmask.shape[0]

# Define base CHD variables

2 CHD packages
1. Surface CHDs (chds in bay, and perimeter chd's in bay ,ALL in the top layer) which only has one external file
2. Perimeter CHDs (perimeter chds not in the bay, at all layers), 366 external files for every day. 

In [ ]:
# chd_all = gwf.get_package("chd_coast")
# chd_data0 = chd_all.stress_period_data.get_dataframe()[0]
# # print_value(chd_data0)
# assert chd_data0.shape[0] == chdmask.shape[0]
# chd_all.stress_period_data.get_dataframe()[0]

# Setup and initialize D-FLOW FM
You will need to set `dflow_dirpath` to the correct directory on your machine.

## Paths

In [ ]:
dflow_dirpath = pl.Path(r"..\dflow-fm\dflowfm_dll").resolve()
dflow_base = pl.Path(r"..\dflow-fm\highres\tides_atm_surge_2018").resolve()
dflow_working = pl.Path(r"..\dflow-fm\highres\tides_atm_surge_2018\run_dflow_only").resolve()
dflow_config = dflow_working / "FlowFM.mdu"
print(dflow_config)

In [ ]:

def ignore_run_dirs(dir, contents):
    return [name for name in contents if "run" in name.lower()]

if dflow_working.is_dir():
    shutil.rmtree(dflow_working)

shutil.copytree(dflow_base, dflow_working,ignore=ignore_run_dirs)

(dflow_working / "output").mkdir(parents=True, exist_ok=True)

In [ ]:
# Add dflowfm dll folder to PATH so that it can be found by the BMIWrapper
os.environ["PATH"] = (
    str(dflow_dirpath) + os.pathsep + os.environ["PATH"]
)

In [ ]:
(pl.Path(dflow_dirpath) / "dflowfm.dll").is_file()

## Initialize D-Flow FM API

In [ ]:
dflowfm = BMIWrapper(
    engine="dflowfm",
    configfile=str(dflow_config),
)

In [ ]:
import os
print("cwd:", os.getcwd())

In [ ]:
dflowfm.initialize()# this line of code is what changes the cwd to  '\dflow-fm\coarse\tides\run'

In [ ]:
import os
print("cwd:", os.getcwd())

## Get data using DFLOW API

In [ ]:
ndxi = int(dflowfm.get_var("ndxi"))
ndx = int(dflowfm.get_var("ndx")) # number of nodes
x = dflowfm.get_var("xz")
y = dflowfm.get_var("yz")
z = dflowfm.get_var("bl")
xy = [(xx, yy) for (xx, yy) in zip(x, y)]

ndx, ndxi, x.shape, y.shape # bnb note: are these values ok? answer: they shouldnt matter because they run with the gp model 

In [ ]:
def read_mdu(path):
    config = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):  # skip comments
                continue
            if "=" in line:
                key, val = [s.strip() for s in line.split("=", 1)]
                config[key.lower()] = val
    return config

mdu_data = read_mdu(dflow_config)
dflow_start_date = mdu_data["refdate"].split()[0]
dflow_start_date = datetime.datetime.strptime(dflow_start_date, "%Y%m%d")
dflow_end_date = dflow_start_date + datetime.timedelta(seconds=dflowfm.get_end_time())
print(f'DFlow start date: {dflow_start_date} \nDFlow end date: {dflow_end_date}')

In [ ]:
qext = np.zeros(ndx)
qext.shape, qext

In [ ]:
qext_cum = np.zeros(ndx)
qext_cum.shape

In [ ]:
vextcum = dflowfm.get_var("vextcum")
vextcum.shape, vextcum

# Initialize MODFLOW using MODFLOW API

## Change Modflow TDIS

In [ ]:
# tdis = sim.get_package("TDIS")
# mf_perioddata= tdis.perioddata.array
# mf6_start_date = datetime.datetime.strptime(tdis.start_date_time.get_data(), "%Y-%m-%d")
# # calculate date for each stress period
# mf_SPdates = [mf6_start_date]
# for perlen, _, _ in mf_perioddata:
#     mf_SPdates.append(mf_SPdates[-1] + pd.Timedelta(days = perlen))
# # remove the last date because it is extra
# mf_SPdates = mf_SPdates[:-1]

# # Compute cumulative total times at the end of each stress period
# mf_totaltimes= tdis.perioddata.array['perlen'].cumsum()

# mf_tdis_df = pd.DataFrame({'Date':mf_SPdates,
#                            'SP_data': list(mf_perioddata),
#                            'SP': [i for i in range(len(mf_perioddata))],
#                            'totim': mf_totaltimes})

In [ ]:

# # latest date among models
# latest_date = max(dflow_start_date, mf6_start_date)
# print("Latest date:", latest_date)

# # find closest available date in mf_tdis_df
# closest_date_idx = (mf_tdis_df['Date'] - latest_date).abs().idxmin() 
# print(closest_date_idx)
# closest_date = mf_tdis_df.loc[closest_date_idx, 'Date']

# # get corresponding stress period
# latest_date_SP = mf_tdis_df.loc[closest_date_idx, 'SP']

# print("Closest available date:", closest_date)
# print("Associated SP:", latest_date_SP)

# print(f'Modflow Stress Period associated with the lastest models start date: {latest_date_SP}')
# assert latest_date_SP + len(mf_perioddata[latest_date_SP:]) == len(mf_tdis_df)
# # Update nstp where SP index >= latest_date_SP
# mf_perioddata["nstp"][latest_date_SP:] = mf_couple_nstp 
# mf_perioddata

# mf_tdis_df['SP_data_coupled'] = list(mf_perioddata)


# # CHANGE TDIS IN MF MODEL TO REFLECT TIME STEP CHANGES
# tdis.mf_perioddata = mf_perioddata
# print(mf_perioddata)


In [ ]:
# sim.write_simulation(silent=silent())

## MF API

In [ ]:
# # set the mdll_path to be the absolute path where the mf6 dll is located
# #mdll_path = pl.Path(r"D:\LISS_GW\GitRepo_CoupledModels\nywsc_compound_flooding\modflow\mf6dll\libmf6.dll")
# mdll_path = str(r"D:\LISS_GW\GitRepo_CoupledModels\nywsc_compound_flooding\modflow\mf6dll\libmf6.dll")
# print(mdll_path)

# mf_run_path = str(mf_run_path)
# print(mf_run_path)

In [ ]:
# mf6 = ModflowApi(mdll_path, working_directory=mf_run_path)

In [ ]:
# mf6.initialize()

# Define Variables and Functions for coupling

## Define MODFLOW variable tags and set pointer to MODFLOW variables

In [ ]:
# ghb_bhead_tag = mf6.get_var_address("BHEAD", "GWF", "GHB")
# ghb_cond_tag = mf6.get_var_address("COND", "GWF", "GHB")
# ghb_flow_tag = mf6.get_var_address("SIMVALS", "GWF", "GHB")

In [ ]:
# ghb_bhead_ptr = mf6.get_value_ptr(ghb_bhead_tag)
# ghb_cond_ptr = mf6.get_value_ptr(ghb_cond_tag)
# ghb_flow = np.zeros(ghb_bhead_ptr.shape)

In [ ]:
# chd_head_tag = mf6.get_var_address("HEAD", "GWF", "CHD_coast")
# chd_flow_tag = mf6.get_var_address("SIMVALS", "GWF", "CHD_coast")

In [ ]:
# chd_head_ptr = mf6.get_value_ptr(chd_head_tag)
# chd_flow = np.zeros(chd_head_ptr.shape)

## Create dictionaries for saving modified GHB data

In [ ]:
# ghb_elev_dict = {}
# ghb_cond_dict = {}
# chd_elev_dict = {}
# qext_dict = {}


## Function to update MODFLOW GHB & CHD data

In [ ]:
# def update_mf(key, s, d):
#     #print('running update_mf')
#     mask = d == 0.0
#     s[mask] = 0.0
#     mult = np.full(d.shape, 1.0)
#     mult[mask] = 0.0

#     ghb_head = ghb_data0["bhead"].to_numpy()
#     # print(f'ghb_data0 : {ghb_data0["bhead"].shape}') 
#     # print(f'ghb_head :{ghb_head.shape}')

#     ghb_head[ghbmask] = dflow2mfghb.dot(s)[ghbmask] * m2ft
#     # print(f'ghbmask : {ghbmask.shape} , dflow2mfghb :{dflow2mfghb.shape}')

#     ghb_cond = ghb_data0["cond"].to_numpy()
#     # print(f'ghb_cond : {ghb_data0["cond"].shape}') 

#     ghb_cond[ghbmask] = ghb_cond[ghbmask] * dflow2mfghb.dot(mult)[ghbmask]
    
#     ghb_bhead_ptr[:] = ghb_head[:] # does not work
#     #ghb_bhead_ptr[:] = ghb_bhead_ptr[:] * 1.5 # _ptr variable is a pointer for the APR. 

#     ghb_cond_ptr[:] = ghb_cond[:] #does not work
#     #ghb_cond_ptr[:] = ghb_cond_ptr[:]
    
#     chd_head = chd_data0["head"].to_numpy()

#     chd_head[chdmask] = dflow2mfchd.dot(s)[chdmask] * m2ft
    
#     chd_head_ptr[:] = chd_head[:] # does not work
#     #chd_head_ptr[:] = chd_head_ptr[:] * 1.5 #this works
   
#     # update results dictionary
#     ghb_elev_dict[key] = ghb_head.copy()
#     ghb_cond_dict[key] = ghb_cond.copy()
#     chd_elev_dict[key] = chd_head.copy()

## Function to update D-Flow FM Qext data

In [ ]:
# def update_dflow(key, d):
#     ghb_flow = -mf6.get_value(ghb_flow_tag) * cfd2cms
#     #print(f'ghb_flow: {ghb_flow.shape, mf6.get_value(ghb_flow_tag).shape}')
    
#     dflow_qext_ghb = ghb2qext.dot(ghb_flow)
#     #print(f'dflow_qext_ghb{dflow_qext_ghb.shape, ghb2qext.shape, ghb_flow.shape}')

#     dflow_qext_ghb[d == 0.0] = 0.0
    
#     chd_flow = -mf6.get_value(chd_flow_tag) * cfd2cms
#     #print(f'chd_flow{chd_flow.shape, mf6.get_value(chd_flow_tag).shape}')

#     dflow_qext_chd = chd2qext.dot(chd_flow)

#     dflow_qext_chd[d == 0.0] = 0.0

#     dflow_qext = dflow_qext_ghb + dflow_qext_chd
    
#     qext_cum[:ndxi] += dflow_qext[:ndxi]
#     qext[:ndxi] = dflow_qext[:ndxi]
#     dflowfm.set_var("qext", qext)

#     # update results dictionaries
#     qext_dict[key] = qext[:ndxi].copy()

# Run each time step

In [ ]:
print(
    f"DFLOWFM current_time: {dflowfm.get_current_time():15,.1f} sec. ({dflowfm.get_current_time()/86400.:15,.1f} days)\n"
     + f"DFLOWFM end_time:     {dflowfm.get_end_time():15,.1f} sec. ({dflowfm.get_end_time()/86400.:15,.1f} days)"
)

In [ ]:
# import datetime
# mf6_start_date = datetime.datetime.strptime(tdis.start_date_time.get_data(), "%Y-%m-%d")
# mf6_end_date = mf6_start_date + datetime.timedelta(days=mf6.get_end_time())
# print(f'MF start date: {mf6_start_date} \nMF end date: {mf6_end_date}')

In [ ]:
print(f'Dflow start date: {dflow_start_date}')
print('-------------------------------------')

idx = 0
jdx = 0
t0 = time.perf_counter()

# NOTE: we'll assume that the modflow model will run the longest
start_date = dflow_start_date
end_date = dflow_end_date


current_date = start_date

while current_date <= dflow_end_date:
    # idx += 1
    # frac_comp = 1 - (dflow_end_date - current_date).days / (dflow_end_date - start_date).days
    # print(f"(Current date: {current_date}) - {frac_comp:6.2%} complete - ({idx:03d})    ", end="\r")
    dflowfm.update()
    current_date = dflow_start_date + datetime.timedelta(seconds=dflowfm.get_current_time())
    print(current_date)
    # if idx == int(dflow_per_mf):
    #     print(f"(*Coupling* - Current date: {current_date}) - {frac_comp:6.2%} complete - ({idx:03d})    ", end="\r")
    #     s = dflowfm.get_var("s1")[:ndxi] # water level
    #     d = dflowfm.get_var("hs")[:ndxi]
        
    #     mf6.prepare_time_step(mf6.get_time_step())
    #     update_mf(str(jdx), s, d)
    #     mf6.do_time_step()
    #     mf6.finalize_time_step()
    #     update_dflow(str(jdx), d)
        
    #     # # advance SWMM
    #     # update_swmm(str(jdx))
    #     # dt_sec = mf6.get_time_step() * d2sec
    #     # swmm_sim.step_advance(int(dt_sec))
    #     # try:
    #     #     swmm_sim.__next__()  
    #     # except StopIteration:
    #     #     break        

    #     # update counters
    #     idx = 0
    #     jdx += 1
    
    if current_date >= dflow_end_date:
        break

vextcum = dflowfm.get_var("vextcum")

t1 = time.perf_counter()
print(f"\nrun time: {(t1 - t0) / 60.} min")

# Results

In [ ]:
dflow_crs = 'EPSG:32618'

In [ ]:
import rasterio
import numpy as np
import xarray as xr
from scipy.interpolate import griddata
import geopandas as gpd
from pathlib import Path
import cartopy.crs as ccrs

In [ ]:
# load the DEM that is in DFLOW CRS
#---------------------------
with rasterio.open(r"C:\Users\bbayrakt\OneDrive - DOI\LISS_GW\GIS\Mosaic_Lidar_20_21.tif\Mosaic_Lidar_20_21_CLIPPED_wgs84utm18.tif") as src:
    x_res, y_res = src.res
    print(f"Resolution: {x_res} x {y_res}")
    dem_meta = src.profile.copy()
    dem_bounds = src.bounds
    dem_arr = src.read(1)
    # create grid x and grid y values for interpolation
    cols, rows = np.meshgrid(np.arange(dem_meta['width']), np.arange(dem_meta['height']))
    grid_xs, grid_ys = rasterio.transform.xy(dem_meta['transform'], rows, cols)

# ASSERT DEM IS IN D FLOW-FM CRS
assert src.crs == dflow_crs

In [ ]:
# load in the unstructured DFLOW grid and filter to time and variables of interest
#-----------------------------------
dflow_nc = xr.open_dataset(dflow_working / "output"/ "FlowFM_map.nc")

In [ ]:
from shapely.geometry import Polygon, LineString
# this allows us to clip a geometry to a bounding box
def clip_to_area(shp, xmin, xmax, ymin, ymax):
    polygon = Polygon([(xmin, ymin), (xmin, ymax), (xmax, ymax), (xmax, ymin), (xmin, ymin)])
    poly_gdf = gpd.GeoDataFrame([1], geometry=[polygon], crs=ccrs.PlateCarree()).to_crs(shp.crs)
    shp_clip = shp.clip(poly_gdf.geometry).to_crs('EPSG:4456')
    return shp_clip


# Create PJ Basemap
#-------------------------
pj_xmin = -73.076
pj_xmax = -73.062
pj_ymin = 40.94
pj_ymax= 40.953

coastal_states = gpd.read_file(r'C:\Users\bbayrakt\OneDrive - DOI\LISS_GW\GW_Models\LISUS_conditionedmodels_BNB\2018\conditioned_model_2018_daily\_mfsetup\GIS\shps\coastal_states.shp')
ny_clip = clip_to_area(coastal_states[coastal_states['STATE_COD'].isin(['NY'])],
                          pj_xmin, pj_xmax, pj_ymin, pj_ymax)

element_crs = ccrs.epsg(4456)


In [ ]:
#================================================
# STEP3 : SELECT DATE+TIME OF INTEREST
#=================================================
t_sel = "2018-09-24 10:30:00" 


dflow_nc_tsel = dflow_nc.sel(time=t_sel, method='nearest')
# Change water levels to be 0 in dry cells, because if water depth is 0, it sets water level to be the land/bed elevation value...
# Where waterdepth >0 (wet cell), keep water level value. Where water depth <0 (dry cell), replace value with 0
dflow_nc_tsel['mesh2d_s1_NEW'] = dflow_nc_tsel['mesh2d_s1'].where(dflow_nc_tsel["mesh2d_waterdepth"] > 0, other=0)

#================================================
# STEP3 : RESAMPLE D FLOW-FM water level RESULTS TO DEM GRID
#=================================================
dflow_waterlevel = griddata(points=(dflow_nc_tsel['mesh2d_face_x'], dflow_nc_tsel['mesh2d_face_y']),
    values=dflow_nc_tsel['mesh2d_s1_NEW'], 
    xi=(grid_xs, grid_ys),
    method='linear')
assert dem_arr.shape == dflow_waterlevel.shape
#=============================================
# STEP 4: CALCULATE AND PLOT WATER DEPTH
#============================================
nodata = -9999.0
# subtract DEM from water levels to get water depth
water_depth = dflow_waterlevel - dem_arr # (numpy array)
# Convert from meters to feet
water_depth = water_depth*3.28084
# Nan water depth values less than 0
water_depth_masked = np.where(water_depth > 0, water_depth, nodata)
# Saving out water depth numpy array as raster
#-----------------------------------
# Create a copy of the DEM metadata
water_meta = dem_meta.copy()
# Update the metadata for the output raster
water_meta.update({
    "dtype": "float32",
    "count": 1,
    "nodata": nodata})
# Save as raster
path = Path.cwd().parent / "GIS" / f"dflow_waterdepth_{t_sel[:10]}.tif"
path.parent.mkdir(parents=True, exist_ok=True)
# explicitly remove existing file
if path.exists():
    path.unlink()
with rasterio.open(path, "w", **water_meta) as dst:
    dst.write(water_depth_masked.astype("float32"), 1)



# ======================================
# STEP 5: PLOT
#==========================================
# load in the water depth raster that was just created and saved out
import rioxarray as rxr
wd = rxr.open_rasterio(path, masked=True).squeeze()
# reproj to basemap
wd_4456 = wd.rio.reproject("EPSG:4456")
assert ny_clip.crs == wd_4456.rio.crs
# Clip Dflow water levels to NY boundary to determine areas of innundation
#--------------------------------------------------------------------------
wd_ny = wd_4456.rio.clip(
    ny_clip.geometry,
    ny_clip.crs,
    drop=True,       )
assert wd_ny.rio.crs == ny_clip.crs



fig = plt.figure(dpi=100,figsize=(5, 7.5))
ax = fig.add_subplot(projection=ccrs.Mercator())
    # Coastal Iundation
#-----------------
ny_clip.plot(ax=ax)
wd_ny.plot(ax=ax,
    add_colorbar=False,
    alpha=0.7,
    transform=element_crs)

